In [1]:
!pip install evaluate
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.3 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
import os
import random
from sklearn.model_selection import train_test_split
from datasets import Dataset
import torch
import transformers
import evaluate
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

In [3]:
df = pd.read_excel("/kaggle/input/datasets/patrickashraf/code-and-translation/Code and Translation.xlsx")

In [4]:
df['Code'].head()

0    D21-:Q3-:D36 F4-:D36 L2-X1-:S19 S29-U23-T21-:X...
1    M17-[?-*A26-*?] S34-/-:Aa1-G43-A1-[?-*Z3-*?] h...
2    W24-:V31 V22-:F34-:N35-M23-X1-:N35 G17 R8-O6-X...
3    G35 F34-F34-F34 D2-*Z1-:Aa17 U6-:D21-M17-M17-X...
4    M17-G43 D4-:N35 M17-M40-O34-:O1-:Z1 G17 V28-*W...
Name: Code, dtype: object

In [5]:
def clean_text_column(col):
    return col.fillna("").astype(str).str.strip()
    
df['Code'] = clean_text_column(df['Code'])
df['Translation'] = clean_text_column(df['Translation'])

In [6]:
def clean_gardiner(text):
    # 1. Remove anything inside square brackets (including brackets)
    text = re.sub(r"\[.*?\]", "", text)

    # 2. Find Gardiner codes:
    # Pattern: letters (1–3) + numbers (e.g., D21, Aa17, Z1, X1, etc.)
    codes = re.findall(r"\b[A-Za-z]{1,3}\d+\b", text)

    return " ".join(codes)

In [7]:
df['Code'] = df['Code'].apply(clean_gardiner)

In [8]:
df["Translation"].fillna("").str.strip().value_counts().head(20)

Translation
Another (remedy):                                                                                 441
                                                                                                  335
Become a mass.                                                                                     86
It will be drunk over 4 days.                                                                      85
(The affected department) will be connected via this.                                              64
I will be squeezed dry.                                                                            60
Get cooked.                                                                                        55
A sacrifice offered by the king, a sacrifice offered by Anubis, who is before the hall of God:     43
Words speak:                                                                                       42
(Will) proceed in the same way.                                       

In [9]:
import pandas as pd
import re

# Work on a copy
df_clean = df.copy()

# Ensure strings
df_clean["Code"] = df_clean["Code"].fillna("").astype(str)
df_clean["Translation"] = df_clean["Translation"].fillna("").astype(str)

def clean_translation(text):
    text = str(text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Remove bracketed damaged/unknown parts inside sentences
    text = re.sub(r"\[\s*\.\.\.\s*\]", " ", text)
    text = re.sub(r"\[\s*---+\s*\]", " ", text)

    # Remove standalone damage markers
    text = re.sub(r"--destroyed--", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bunknown\b", " ", text, flags=re.IGNORECASE)

    # Remove repeated dashes / ellipses anywhere
    text = re.sub(r"\.{2,}", " ", text)
    text = re.sub(r"-{2,}", " ", text)

    # Remove empty square brackets left behind
    text = re.sub(r"\[\s*\]", " ", text)

    # Normalize quotation marks
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("’", "'").replace("‘", "'")

    # Remove extra spaces before punctuation
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)

    # Normalize spaces again
    text = re.sub(r"\s+", " ", text).strip()

    return text


df_clean["Translation_clean"] = df_clean["Translation"].apply(clean_translation)

# Remove rows only if translation became empty or useless
bad_exact = {
    "",
    ".",
    ",",
    ";",
    ":",
    "-",
    "...",
    "[...]",
    "[---]",
}

df_clean = df_clean[
    ~df_clean["Translation_clean"].str.strip().isin(bad_exact)
]

# Remove rows with no Gardiner code
df_clean = df_clean[df_clean["Code"].str.strip().ne("")]

# Optional: remove very short targets that are likely fragments
# Use this carefully; you may comment it out if short translations are valid.
df_clean = df_clean[
    df_clean["Translation_clean"].str.split().str.len() >= 2
]

# Replace original column
df_clean["Translation"] = df_clean["Translation_clean"]
df_clean = df_clean.drop(columns=["Translation_clean"])

# Remove exact duplicate input-output pairs
df_clean = df_clean.drop_duplicates(subset=["Code", "Translation"])

# Reset index
df_clean = df_clean.reset_index(drop=True)

print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("Removed rows:", len(df) - len(df_clean))

df_clean.head(20)

Original rows: 38021
Cleaned rows: 33220
Removed rows: 4801


,Code,Translation
0,D21 Q3 D36 F4 D36 L2 X1 S19 S29 U23 T21 X1 G17...,"Hereditary noble and prince, royal seal-bearer..."
1,M17 S34 Aa1 G43 A1 N17 A1 S29 V4 X1 S29 N35 D2...,"O living ones, who are upon the earth, who sha..."
2,G35 F34 F34 F34 D2 Z1 Aa17 U6 D21 M17 M17 X1 N...,"A trusted one upon the landing place,great ove..."
3,M17 G43 D4 N35 M17 M40 O34 O1 Z1 G17 V28 W14 X...,"I built a tomb through the favour of the king,..."
4,M17 G43 N35 O4 Q3 Y2 Z2 W24 Z1 M17 M2 O34 X8 F...,"I restored the laws of the ancient times, it w..."
5,M17 G43 D46 O4 N35 D1 N35 X1 A1 Z2 D21 D28 X1 ...,I employed a group of artisans for the work in...
6,D35 D21 D37 G1 W11 A1 D21 D46 D58 V28 F18 X1 A...,I was not let to suffer lack in the treasury w...
7,Aa1 N35 X1 N37 D54 M17 W19 Z1 S29 D21 V30 N35 ...,His Majesty caused that I led a pleasant life ...
8,W24 V31 Y5 Aa1 U22 D21 Aa13 Z1 V30 I9 S29 Y5 N...,"I was one efficient at the side of his master,..."
9,D50 G17 D52 Y2 H6 G43 G17 W11 D21 W11 U17 A2 D...,"I was one straightforward in the presence, fre..."


In [10]:
train_df, temp_df = train_test_split(df_clean, test_size=0.2, random_state=42)

val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [11]:
train_df["input_text"] = "translate MdC to English: " + train_df["Code"]
val_df["input_text"] = "translate MdC to English: " + val_df["Code"]
test_df["input_text"] = "translate MdC to English: " + test_df["Code"]

train_df["target_text"] = train_df["Translation"]
val_df["target_text"] = val_df["Translation"]
test_df["target_text"] = test_df["Translation"]

train_dataset = Dataset.from_pandas(train_df[["input_text", "target_text"]])
val_dataset = Dataset.from_pandas(val_df[["input_text", "target_text"]])
test_dataset = Dataset.from_pandas(test_df[["input_text", "target_text"]])

train_dataset = train_dataset.remove_columns(["__index_level_0__"])
val_dataset = val_dataset.remove_columns(["__index_level_0__"])
test_dataset = test_dataset.remove_columns(["__index_level_0__"])

In [12]:
model_name = "facebook/m2m100_418M"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

tokenizer.src_lang = "en"
tokenizer.tgt_lang = "en"

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

In [13]:
max_input_length = 512
max_target_length = 128

def preprocess_function(batch):
    inputs = [str(x).strip() for x in batch["input_text"]]
    targets = [str(x).strip() for x in batch["target_text"]]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

val_tokenized = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names
)


test_tokenized = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

train_tokenized

Map:   0%|          | 0/26576 [00:00<?, ? examples/s]

Map:   0%|          | 0/3322 [00:00<?, ? examples/s]

Map:   0%|          | 0/3322 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26576
})

In [14]:
# after tokenization
sample = train_tokenized[0]

print("INPUT IDS:", sample["input_ids"][:30])
print("LABEL IDS:", sample["labels"][:30])

print("Decoded input:")
print(tokenizer.decode(sample["input_ids"], skip_special_tokens=False))

valid_labels = [x for x in sample["labels"] if x != -100]
print("Valid label token count:", len(valid_labels))

print("Decoded label:")
print(tokenizer.decode(valid_labels, skip_special_tokens=False))

INPUT IDS: [128022, 5815, 80447, 100, 173, 247, 128, 18006, 9, 159, 4893, 159, 7351, 129, 451, 105, 21328, 765, 451, 315, 465, 203, 718, 177, 451, 177, 6227, 158, 622, 68]
LABEL IDS: [128022, 33, 36127, 72886, 13635, 1381, 66954, 4398, 409, 1766, 203, 5, 434, 5, 480, 5, 15592, 14810, 14291, 117, 1307, 72886, 13635, 1381, 66954, 1197, 244, 49, 12524, 432]
Decoded input:
__en__translate MdC to English: N35 N37 K1 D54 X1 Z4 I9 O1 O29 G7 S34 G17 Q1 X1 O1 I9 X1 N35 N35 N37 K1 D54 X1 Z4 I9 Aa1 I9 X1 Z6 N35 G5 G7 D2 Z1 V22 N35 V28 X1 I12 I9</s>
Valid label token count: 42
Decoded label:
__en__"He who will displace Pharaoh I.H.G. from his place is he who will displace the 'enemy of Horus' with his forehead serpent!"</s>


In [15]:
bad = 0
lengths = []

for i in range(len(train_tokenized)):
    labels = train_tokenized[i]["labels"]
    valid = [x for x in labels if x != -100]
    lengths.append(len(valid))
    if len(valid) == 0:
        bad += 1

print("Bad label rows:", bad)
print("Min label length:", min(lengths))
print("Max label length:", max(lengths))
print("Average label length:", sum(lengths) / len(lengths))

Bad label rows: 0
Min label length: 4
Max label length: 128
Average label length: 27.516593919325707


In [16]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

training_args = Seq2SeqTrainingArguments(
    output_dir="./results_byt5_clean",

    eval_strategy="epoch",
    save_strategy="no",

    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=5,

    logging_steps=50,
    logging_strategy="steps",

    fp16=False,
    bf16=False,

    predict_with_generate=False,
    report_to="none",

    remove_unused_columns=True,

    max_grad_norm=1.0
)

In [17]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator
)

In [18]:
batch = data_collator([train_tokenized[i] for i in range(4)])
batch = {k: v.to(model.device) for k, v in batch.items()}

with torch.no_grad():
    print("Train batch loss:", model(**batch).loss.item())

batch = data_collator([val_tokenized[i] for i in range(4)])
batch = {k: v.to(model.device) for k, v in batch.items()}

with torch.no_grad():
    print("Val batch loss:", model(**batch).loss.item())

Train batch loss: 4.638047695159912
Val batch loss: 5.193603515625


In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.462478,2.145468
2,1.755900,1.826000
3,1.304365,1.686052
4,0.995642,1.628502
5,0.701385,1.622445


TrainOutput(global_step=33220, training_loss=1.5738831558549067, metrics={'train_runtime': 15232.0756, 'train_samples_per_second': 8.724, 'train_steps_per_second': 2.181, 'total_flos': 3.695226165588787e+16, 'train_loss': 1.5738831558549067, 'epoch': 5.0})

In [20]:
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")


sample_test = test_df.copy()

pred_texts = []
true_texts = sample_test["target_text"].tolist()
input_texts = sample_test["input_text"].tolist()

model.eval()

for text in tqdm(input_texts):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=max_target_length,
            num_beams=4,
            early_stopping=True
        )

    pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    pred_texts.append(pred.strip())

true_texts = [t.strip() for t in true_texts]

bleu_result = bleu.compute(
    predictions=pred_texts,
    references=[[t] for t in true_texts]
)

rouge_result = rouge.compute(
    predictions=pred_texts,
    references=true_texts
)

exact_match = np.mean([int(p == t) for p, t in zip(pred_texts, true_texts)])

print({
    "bleu": bleu_result["score"],
    "rouge1": rouge_result["rouge1"],
    "rouge2": rouge_result["rouge2"],
    "rougeL": rouge_result["rougeL"],
    "exact_match": exact_match
})

  0%|          | 0/3322 [00:00<?, ?it/s]

{'bleu': 26.816075092747138, 'rouge1': np.float64(0.45897117615168304), 'rouge2': np.float64(0.2891596221504299), 'rougeL': np.float64(0.4416527307296584), 'exact_match': np.float64(0.04304635761589404)}


In [21]:
save_path = "/kaggle/working/byt5_small_mdc_translation"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Saved to:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /kaggle/working/byt5_small_mdc_translation


In [22]:
custom_text = "translate MdC to English: D21 Q3 D36 F4 D36"

inputs = tokenizer(
    custom_text,
    return_tensors="pt",
    truncation=True,
    max_length=max_input_length
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_length=max_target_length,
        num_beams=4,
        early_stopping=True
    )

prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("Prediction:", prediction)

Prediction: Hereditary noble and local prince,


In [23]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

def semantic_similarity(preds, refs):
    pred_emb = model.encode(preds)
    ref_emb = model.encode(refs)
    
    sims = cosine_similarity(pred_emb, ref_emb)
    return sims.diagonal().mean()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
semantic_result = semantic_similarity(
    pred_texts,
    true_texts
)
semantic_result

np.float32(0.5875533)